<a href="https://colab.research.google.com/github/safarashwan2005/w1/blob/checking-answer-of-schr.-equation/w1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:

!pip uninstall -y jax jaxlib jax_cuda12_plugin jax_cuda12_pjrt dynamiqs


!pip install -U "jax[cuda12]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

!pip install dynamiqs

Found existing installation: jax 0.11.2
Uninstalling jax-0.11.2:
  Successfully uninstalled jax-0.11.2
Found existing installation: jaxlib 0.11.2
Uninstalling jaxlib-0.11.2:
  Successfully uninstalled jaxlib-0.11.2
Found existing installation: jax-cuda12-plugin 0.11.2
Uninstalling jax-cuda12-plugin-0.11.2:
  Successfully uninstalled jax-cuda12-plugin-0.11.2
Found existing installation: jax-cuda12-pjrt 0.11.2
Uninstalling jax-cuda12-pjrt-0.11.2:
  Successfully uninstalled jax-cuda12-pjrt-0.11.2
Found existing installation: dynamiqs 0.3.6
Uninstalling dynamiqs-0.3.6:
  Successfully uninstalled dynamiqs-0.3.6
Looking in links: https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
  Using cached jax-0.11.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.11.2-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (1.3 kB)
  Using cached jax_cuda12_plugin-0.11.2-cp313-cp313-manylinux_2_27_x86_64.whl.metadata (2.0 kB)
  Using cached jax_cuda12_pjrt-0.11.2-py3-none-manylinux_2_2

  Using cached dynamiqs-0.3.6-py3-none-any.whl.metadata (14 kB)
Using cached dynamiqs-0.3.6-py3-none-any.whl (175 kB)


In [1]:

!pip uninstall -y jax jaxlib jax-cuda12-pjrt jax-cuda12-plugin jax-cuda13-pjrt jax-cuda13-plugin dynamiqs


!pip install --no-cache-dir -U "jax[cuda12]" dynamiqs

Found existing installation: jax-cuda12-pjrt 0.11.2
Uninstalling jax-cuda12-pjrt-0.11.2:
  Successfully uninstalled jax-cuda12-pjrt-0.11.2
Found existing installation: jax-cuda12-plugin 0.11.2
Uninstalling jax-cuda12-plugin-0.11.2:
  Successfully uninstalled jax-cuda12-plugin-0.11.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.2/188.2 MB 215.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 MB 184.4 MB/s eta 0:00:00


In [2]:
import jax
print(jax.devices())

[CudaDevice(id=0)]


In [3]:
!pip uninstall -y dynamiqs
!pip install git+https://github.com/dynamiqs/dynamiqs.git

Found existing installation: dynamiqs 0.3.6
Uninstalling dynamiqs-0.3.6:
  Successfully uninstalled dynamiqs-0.3.6
  Cloning https://github.com/dynamiqs/dynamiqs.git to /tmp/pip-req-build-xzx103vn
  Running command git clone --filter=blob:none --quiet https://github.com/dynamiqs/dynamiqs.git /tmp/pip-req-build-xzx103vn
  Resolved https://github.com/dynamiqs/dynamiqs.git to commit 58a6baa385f6824fe8ac107b9b26935f19ee6bc6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dynamiqs: filename=dynamiqs-0.3.6-py3-none-any.whl size=193879 sha256=7e77a2e5333d530876993763ce540d01f482fbb0f6d17039b55a2928543b94cd
  Stored in directory: /tmp/pip-ephem-wheel-cache-3f0fcaov/wheels/37/b9/05/dad2d56401030647e7026c40b94de962b93e223879c99b1ab8
Successfully built dynamiqs


In [4]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"

import warnings
import time
import numpy as np
import scipy.sparse as sp
from scipy.sparse import kron as sp_kron

import jax
import jax.numpy as jnp
import dynamiqs as dq

jax.config.update("jax_enable_x64", True)
dq.set_precision("double")

print("Available JAX Devices:", jax.devices())
warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)

class NVCenterNearestNeighborDQ:
    def __init__(self, params):
        self.N = params['N']
        self.D = params['D']
        self.omega_0 = params['omega_0']
        self.gamma_e = params['gamma_e']
        self.Bz = params['Bz']
        self.zeta = params['zeta']
        self.g = params['g']


        self.I3 = dq.eye(3)
        self.Sz = jnp.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=jnp.complex128)
        self.Sx = (1 / jnp.sqrt(2)) * jnp.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=jnp.complex128)
        self.Sy_prime = (1 / jnp.sqrt(2)) * jnp.array([[0, -1j, 0], [1j, 0, -1j], [0, 1j, 0]], dtype=jnp.complex128)

    def _expand_operator_dense(self, op, target_index):
        result = op if target_index == 0 else self.I3
        for i in range(1, self.N):
            current_op = op if i == target_index else self.I3
            result = dq.tensor(result, current_op)
        return result

    def get_control_hamiltonian(self):
        dim = 3**self.N
        H_control = jnp.zeros((dim, dim), dtype=jnp.complex128)
        cos_z, sin_z = jnp.cos(self.zeta), jnp.sin(self.zeta)

        for j in range(self.N):
            Sx_j = self._expand_operator_dense(self.Sx, j)
            Sy_p_j = self._expand_operator_dense(self.Sy_prime, j)
            H_control = H_control + 0.5 * (cos_z * Sx_j + sin_z * Sy_p_j)
        return H_control

    def get_drift_hamiltonian(self):
        dim = 3**self.N
        H_drift = jnp.zeros((dim, dim), dtype=jnp.complex128)
        Sz_sq = self.Sz @ self.Sz

        for j in range(self.N):
            Sz_j_sq = self._expand_operator_dense(Sz_sq, j)
            Sz_j = self._expand_operator_dense(self.Sz, j)
            term1 = (self.D[j] - self.omega_0) * Sz_j_sq
            term2 = self.gamma_e * self.Bz[j] * Sz_j
            H_drift = H_drift + (term1 - term2)

        for j in range(self.N - 1):
            Sz_j = self._expand_operator_dense(self.Sz, j)
            Sz_j_next = self._expand_operator_dense(self.Sz, j + 1)
            H_drift = H_drift + self.g[j] * (Sz_j @ Sz_j_next)

        return H_drift

    def get_ground_state(self):
        ms_0 = jnp.array([[0], [1], [0]], dtype=jnp.complex128)
        ground_state = ms_0
        for _ in range(1, self.N):
            ground_state = dq.tensor(ground_state, ms_0)
        return ground_state

    def evolve_state(
        self, initial_state, t_total, control_pulse_func, time_steps=2000
    ):


        H_drift = self.get_drift_hamiltonian()
        H_control = self.get_control_hamiltonian()

        H_t = H_drift + dq.modulated(control_pulse_func, H_control)

        tsave = jnp.array([0.0, t_total]) ####eh elli et8yr? w kaman el jnp.complex128 hat2asar?
        dt = t_total / time_steps

        result = dq.sesolve(H_t, initial_state, tsave, method=dq.method.Euler(dt=dt))

        # Robust unboxing across different dynamiqs versions
        final_state = result.states[-1]
        if hasattr(final_state, 'to_jax'):
            return final_state.to_jax()
        else:
            return dq.to_jax(final_state) if hasattr(dq, 'to_jax') else jnp.asarray(final_state)


# ==========================================
# Automated Execution Block
# ==========================================
if __name__ == "__main__":

    NUM_RUNS = 3
    n_range = range(1, 9)  # Loop from N=1 to N=8 inclusive

    # Dictionary to track execution times for each N across all runs
    times_per_N = {N: [] for N in n_range}

    total_start_time = time.perf_counter()

    # Outer loop for the 10 runs
    for run in range(1, NUM_RUNS + 1):
        print(f"\n{'#'*60}")
        print(f"                   STARTING RUN {run}/{NUM_RUNS}                   ")
        print(f"{'#'*60}")

        # Inner loop for N=1 to N=8
        for N_test in n_range:
            iter_start_time = time.perf_counter()

            print(f"\n{'='*50}")
            print(f" [RUN {run}] SIMULATING FOR N = {N_test} NV CENTERS ")
            print(f"{'='*50}")

            test_params = {
                'N': N_test,
                'D': [2870.0] * N_test,
                'omega_0': 2870.0,
                'gamma_e': 28000.0,
                'Bz': [1e-3] * N_test,
                'zeta': np.pi / 4,
                'g': [0.1] * (N_test - 1),
            }

            print(f"Building Hamiltonians for N={N_test}...")
            nv_chain = NVCenterNearestNeighborDQ(test_params)
            initial_state_cpu = nv_chain.get_ground_state()
            t_pulse = 1.0

            print("\n--- Testing Lambda Function Pulse ---")
            def lambda_pulse(t):
                return 2.0 * jnp.sin(jnp.pi * (t / t_pulse))

            final_state_lambda = nv_chain.evolve_state(
                initial_state=initial_state_cpu,
                t_total=t_pulse,
                control_pulse_func=lambda_pulse,
                time_steps=2000,
            )

            norm_lambda = jnp.linalg.norm(final_state_lambda)
            print("Norm check (Lambda):", norm_lambda)



            iter_end_time = time.perf_counter()
            elapsed_time = iter_end_time - iter_start_time

            # Save the elapsed time for this specific N to calculate the average later
            times_per_N[N_test].append(elapsed_time)
            if run == 1:
              np.save(f"dq_state_N{N_test}.npy", np.asarray(final_state_lambda).ravel())
              print("dtype check (should be complex128):", final_state_lambda.dtype)

            print(f"\nTime taken for N={N_test} (Run {run}): {elapsed_time:.4f} seconds")

    total_end_time = time.perf_counter()

    # Print total execution time
    print(f"\n{'='*50}")
    print(f" TOTAL TIME FOR ALL {NUM_RUNS} RUNS: {total_end_time - total_start_time:.4f} seconds ")
    print(f"{'='*50}")

    # Print the calculated averages for each N
    print(f"\n{'*'*50}")
    print(f" AVERAGE EXECUTION TIME PER N OVER {NUM_RUNS} RUNS ")
    print(f"{'*'*50}")
    for N_test in n_range:
        avg_time = np.mean(times_per_N[N_test])
        print(f"N = {N_test}: {avg_time:.4f} seconds")
    print(f"{'*'*50}\n")

Available JAX Devices: [CudaDevice(id=0)]

############################################################
                   STARTING RUN 1/3                   
############################################################

 [RUN 1] SIMULATING FOR N = 1 NV CENTERS 
Building Hamiltonians for N=1...

--- Testing Lambda Function Pulse ---


|██████████| 100.0% ◆ elapsed 835.46ms ◆ remaining 0.00ms


Norm check (Lambda): 1.0000056641053718
dtype check (should be complex128): complex128

Time taken for N=1 (Run 1): 13.9047 seconds

 [RUN 1] SIMULATING FOR N = 2 NV CENTERS 
Building Hamiltonians for N=2...

--- Testing Lambda Function Pulse ---


|██████████| 100.0% ◆ elapsed 397.35ms ◆ remaining 0.00ms


Norm check (Lambda): 1.0000113283779832
dtype check (should be complex128): complex128

Time taken for N=2 (Run 1): 7.2917 seconds

 [RUN 1] SIMULATING FOR N = 3 NV CENTERS 
Building Hamiltonians for N=3...

--- Testing Lambda Function Pulse ---


...

Norm check (Lambda): 1.0000339870935337

Time taken for N=6 (Run 3): 5.6445 seconds

 [RUN 3] SIMULATING FOR N = 7 NV CENTERS 
Building Hamiltonians for N=7...

--- Testing Lambda Function Pulse ---


|██████████| 100.0% ◆ elapsed 2.12s ◆ remaining 0.00ms 


Norm check (Lambda): 1.0000396521787605

Time taken for N=7 (Run 3): 8.5794 seconds

 [RUN 3] SIMULATING FOR N = 8 NV CENTERS 
Building Hamiltonians for N=8...

--- Testing Lambda Function Pulse ---


|██████████| 100.0% ◆ elapsed 15.28s ◆ remaining 0.00ms

Norm check (Lambda): 1.000045317426544

Time taken for N=8 (Run 3): 70.4213 seconds

 TOTAL TIME FOR ALL 3 RUNS: 411.1368 seconds 

**************************************************
 AVERAGE EXECUTION TIME PER N OVER 3 RUNS 
**************************************************
N = 1: 7.5346 seconds
N = 2: 5.2649 seconds
N = 3: 4.9335 seconds
N = 4: 5.0560 seconds
N = 5: 5.1626 seconds
N = 6: 5.8639 seconds
N = 7: 9.7802 seconds
N = 8: 93.4477 seconds
**************************************************



In [8]:
import warnings
import numpy as np
import scipy.sparse as sp
from scipy.sparse import kron as sp_kron
import cupy as cp
import cupyx.scipy.sparse as cp_sp
import time
import gc
start_time = time.perf_counter()

warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)



class NVCenterSystemBaseGPU:

    def __init__(self, params):
        self.N = params['N']
        self.D = params['D']
        self.omega_0 = params['omega_0']
        self.gamma_e = params['gamma_e']
        self.Bz = params['Bz']
        self.zeta = params['zeta']
        self.g = params['g']

        # Single-spin operators constructed on CPU to avoid GPU VRAM exhaustion during expansion
        self.I3 = sp.eye(3, dtype=complex, format='csr')
        self.Sz = sp.csr_matrix(
            np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=complex)
        )
        self.Sx = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=complex)
        )
        self.Sy_prime = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, -1j, 0], [1j, 0, -1j], [0, 1j, 0]], dtype=complex)
        )

    def _expand_operator_sparse(self, op, target_index):
        """Expands a 3x3 operator onto the N-spin Hilbert space in CPU system memory."""
        result = op if target_index == 0 else self.I3
        for i in range(1, self.N):
            current_op = op if i == target_index else self.I3
            result = sp_kron(result, current_op, format='csr')
        return result

    def get_control_hamiltonian(self):
        dim = 3**self.N
        H_control = sp.csr_matrix((dim, dim), dtype=complex)
        cos_z = np.cos(self.zeta)
        sin_z = np.sin(self.zeta)

        for j in range(self.N):
            Sx_j = self._expand_operator_sparse(self.Sx, j)
            Sy_p_j = self._expand_operator_sparse(self.Sy_prime, j)
            H_control = H_control + 0.5 * (cos_z * Sx_j + sin_z * Sy_p_j)

        return H_control

    def get_ground_state(self):
        ms_0 = np.array([0, 1, 0], dtype=complex)
        ground_state = ms_0
        for _ in range(1, self.N):
            ground_state = np.kron(ground_state, ms_0)
        return ground_state

    def _euler_solve_optimized(self, H_drift, H_control, u_array, y0, t_total, num_steps):
        # Executes this integration on GPU.
        dt = t_total / num_steps
        c = -1j
        psi = y0.copy()

        for i in range(num_steps):
            u_t = u_array[i]
            k1 = c * (H_drift.dot(psi) + u_t * H_control.dot(psi))
            psi += dt * k1

        return psi

    def evolve_state_optimized(self, initial_state, t_total, control_pulse, time_steps=2000):
        if not callable(control_pulse):
            raise TypeError("control_pulse must be a callable function (e.g., a lambda).")

        H_drift_cpu = self.get_drift_hamiltonian()
        H_control_cpu = self.get_control_hamiltonian()

        H_drift = cp_sp.csr_matrix(H_drift_cpu)
        H_control = cp_sp.csr_matrix(H_control_cpu)

        initial_state_gpu = cp.asarray(initial_state).reshape(-1, 1)

        # Generate timestamps for Euler (1 per step)
        euler_time_points = cp.linspace(0, t_total, time_steps, endpoint=False)

        t_np = cp.asnumpy(euler_time_points)
        u_array = cp.asarray(control_pulse(t_np))

        return self._euler_solve_optimized(
            H_drift, H_control, u_array, initial_state_gpu, t_total, num_steps=time_steps
        )
class NVCenterNearestNeighborGPU(NVCenterSystemBaseGPU):

    def get_drift_hamiltonian(self):
        dim = 3**self.N
        H_drift = sp.csr_matrix((dim, dim), dtype=complex)

        Sz_sq = self.Sz.dot(self.Sz)

        for j in range(self.N):
            Sz_j_sq = self._expand_operator_sparse(Sz_sq, j)
            Sz_j = self._expand_operator_sparse(self.Sz, j)

            term1 = (self.D[j] - self.omega_0) * Sz_j_sq
            term2 = self.gamma_e * self.Bz[j] * Sz_j
            H_drift = H_drift + (term1 - term2)

        for j in range(self.N - 1):
            Sz_j = self._expand_operator_sparse(self.Sz, j)
            Sz_j_next = self._expand_operator_sparse(self.Sz, j + 1)
            H_drift = H_drift + self.g[j] * Sz_j.dot(Sz_j_next)

        return H_drift


# =======================================
# Execution Block
# ==========================================
if __name__ == "__main__":
    NUM_RUNS = 3
    n_range = range(1, 9)


    times_per_N = {N: [] for N in n_range}

    for run in range(1, NUM_RUNS + 1):
        print(f"\n========================================")
        print(f"             STARTING RUN {run}/{NUM_RUNS}             ")
        print(f"========================================")

        for N_test in n_range:

            iter_start_time = time.perf_counter()

            print(f"\n{'='*40}")
            print(f" [RUN {run}/{NUM_RUNS}] SIMULATING FOR N = {N_test} NV CENTERS ")
            print(f"{'='*40}")

            test_params = {
                'N': N_test,
                'D': [2870.0] * N_test,
                'omega_0': 2870.0,
                'gamma_e': 28000.0,
                'Bz': [1e-3] * N_test,
                'zeta': np.pi / 4,
                'g': [0.1] * (N_test - 1),
            }

            print("Building Sparse Hamiltonians on CPU...")
            nv_chain = NVCenterNearestNeighborGPU(test_params)
            initial_state_cpu = nv_chain.get_ground_state()
            t_pulse = 1.0

            print("--- Testing Lambda Function Pulse ---")
            lambda_pulse = lambda t: 2.0 * np.sin(np.pi * (t / t_pulse))

            final_state_lambda = nv_chain.evolve_state_optimized(
                initial_state=initial_state_cpu,
                t_total=t_pulse,
                control_pulse=lambda_pulse,
                time_steps=2000,
            )

            norm_lambda = cp.asnumpy(cp.linalg.norm(final_state_lambda))
            print("Norm check (Lambda):", norm_lambda)


            cp.cuda.Stream.null.synchronize()
            iter_end_time = time.perf_counter()
            elapsed_time = iter_end_time - iter_start_time


            times_per_N[N_test].append(elapsed_time)

            print(f"\n[TIME TAKEN FOR N = {N_test} (Run {run})]: {elapsed_time:.4f} seconds")
            if run == 1:
              np.save(f"cupy_state_N{N_test}.npy", cp.asnumpy(final_state_lambda).ravel())

            # Free GPU memory and CPU garbage collection
            del nv_chain, initial_state_cpu, final_state_lambda
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()

    end_time = time.perf_counter()

    print(f"\n{'='*50}")
    print(f"TOTAL EXECUTION TIME (ALL {NUM_RUNS} RUNS): {end_time - start_time:.4f} seconds")
    print(f"{'='*50}")

    print(f"\n{'='*50}")
    print(f" AVERAGE EXECUTION TIME PER N OVER {NUM_RUNS} RUNS ")
    print(f"{'='*50}")
    for N_test in n_range:
        avg_seconds = np.mean(times_per_N[N_test])
        std_seconds = np.std(times_per_N[N_test])
        print(f"N = {N_test:2d} | Avg Time: {avg_seconds:.4f} s (±{std_seconds:.4f} s)")
    print(f"{'='*50}")




             STARTING RUN 1/3             

 [RUN 1/3] SIMULATING FOR N = 1 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.0000056641053718

[TIME TAKEN FOR N = 1 (Run 1)]: 0.7445 seconds

 [RUN 1/3] SIMULATING FOR N = 2 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.000011328377983

[TIME TAKEN FOR N = 2 (Run 1)]: 0.6888 seconds

 [RUN 1/3] SIMULATING FOR N = 3 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.000016992813091

[TIME TAKEN FOR N = 3 (Run 1)]: 0.6832 seconds

 [RUN 1/3] SIMULATING FOR N = 4 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.0000226574107114

[TIME TAKEN FOR N = 4 (Run 1)]: 0.6908 seconds

 [RUN 1/3] SIMULATING FOR N = 5 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse 

In [9]:
!ls *.npy

cupy_state_N1.npy  cupy_state_N5.npy  dq_state_N1.npy  dq_state_N5.npy
cupy_state_N2.npy  cupy_state_N6.npy  dq_state_N2.npy  dq_state_N6.npy
cupy_state_N3.npy  cupy_state_N7.npy  dq_state_N3.npy  dq_state_N7.npy
cupy_state_N4.npy  cupy_state_N8.npy  dq_state_N4.npy  dq_state_N8.npy


In [10]:
import numpy as np

print(f"{'N':>2} | {'max|diff|':>10} | {'rel L2 err':>10} | {'1-fidelity':>10} | {'norm dq':>9} | {'norm cupy':>9}")
for N in range(1, 9):
    a = np.load(f"dq_state_N{N}.npy").astype(np.complex128)
    b = np.load(f"cupy_state_N{N}.npy").astype(np.complex128)

    max_diff = np.max(np.abs(a - b))
    rel_l2   = np.linalg.norm(a - b) / np.linalg.norm(b)
    fid      = np.abs(np.vdot(a, b))**2 / (np.linalg.norm(a)**2 * np.linalg.norm(b)**2)

    print(f"{N:2d} | {max_diff:10.2e} | {rel_l2:10.2e} | {1-fid:10.2e} | "
          f"{np.linalg.norm(a):9.6f} | {np.linalg.norm(b):9.6f}")

 N |  max|diff| | rel L2 err | 1-fidelity |   norm dq | norm cupy
 1 |   4.85e-17 |   6.87e-17 |   0.00e+00 |  1.000006 |  1.000006
 2 |   2.22e-16 |   2.33e-16 |   0.00e+00 |  1.000011 |  1.000011
 3 |   6.96e-17 |   1.14e-16 |  -8.88e-16 |  1.000017 |  1.000017
 4 |   4.63e-17 |   8.87e-17 |  -8.88e-16 |  1.000023 |  1.000023
 5 |   7.84e-17 |   1.51e-16 |  -8.88e-16 |  1.000028 |  1.000028
 6 |   2.22e-16 |   2.79e-16 |  -4.44e-16 |  1.000034 |  1.000034
 7 |   4.44e-16 |   4.74e-16 |  -1.78e-15 |  1.000040 |  1.000040
 8 |   6.66e-16 |   7.03e-16 |   0.00e+00 |  1.000045 |  1.000045


In [11]:
!ls *.npy

cupy_state_N1.npy  cupy_state_N5.npy  dq_state_N1.npy  dq_state_N5.npy
cupy_state_N2.npy  cupy_state_N6.npy  dq_state_N2.npy  dq_state_N6.npy
cupy_state_N3.npy  cupy_state_N7.npy  dq_state_N3.npy  dq_state_N7.npy
cupy_state_N4.npy  cupy_state_N8.npy  dq_state_N4.npy  dq_state_N8.npy


In [14]:
import os
import numpy as np
import scipy.sparse as sp
from scipy.integrate import solve_ivp

# ---- Same parameters as two scripts ----
D, omega_0, gamma_e, Bz, zeta, g = 2870.0, 2870.0, 28000.0, 1e-3, np.pi/4, 0.1
T = 1.0
u = lambda t: 2.0 * np.sin(np.pi * t / T)      # the control pulse

N_LIST = [1, 2, 3, 4, 5]
STEPS  = [2000, 4000, 8000, 16000]

# ---- Single-spin operators ----
Sz  = np.diag([1, 0, -1]).astype(complex)
Sx  = np.array([[0,1,0],[1,0,1],[0,1,0]], dtype=complex) / np.sqrt(2)
Syp = np.array([[0,-1j,0],[1j,0,-1j],[0,1j,0]], dtype=complex) / np.sqrt(2)
I3  = np.eye(3, dtype=complex)

def op_on(op, j, N):
    out = sp.csr_matrix(op if j == 0 else I3)
    for i in range(1, N):
        out = sp.kron(out, sp.csr_matrix(op if i == j else I3), format="csr")
    return out

def build_H(N):
    dim = 3**N
    H0 = sp.csr_matrix((dim, dim), dtype=complex)
    Hc = sp.csr_matrix((dim, dim), dtype=complex)
    Sz2 = Sz @ Sz
    for j in range(N):
        H0 = H0 + (D - omega_0) * op_on(Sz2, j, N) - gamma_e * Bz * op_on(Sz, j, N)
        Hc = Hc + 0.5 * (np.cos(zeta) * op_on(Sx, j, N) + np.sin(zeta) * op_on(Syp, j, N))
    for j in range(N - 1):
        H0 = H0 + g * (op_on(Sz, j, N) @ op_on(Sz, j + 1, N))
    return H0.tocsr(), Hc.tocsr()

def start_state(N):
    ms0 = np.array([0, 1, 0], dtype=complex)
    psi = ms0
    for _ in range(1, N):
        psi = np.kron(psi, ms0)
    return psi

def euler(H0, Hc, psi0, steps):
    dt = T / steps
    psi = psi0.copy()
    for i in range(steps):
        t = i * dt
        psi = psi + dt * (-1j) * (H0 @ psi + u(t) * (Hc @ psi))
    return psi

def reference(H0, Hc, psi0):
    f = lambda t, y: -1j * (H0 @ y + u(t) * (Hc @ y))
    sol = solve_ivp(f, [0, T], psi0, method="DOP853", rtol=1e-10, atol=1e-12)
    return sol.y[:, -1]

def rel_err(a, b):
    return np.linalg.norm(a - b) / np.linalg.norm(b)

def load(name):
    return np.load(name).astype(complex).ravel() if os.path.exists(name) else None

fmt = lambda x: "   (missing)" if x is None else f"{x:12.3e}"

table1, table2 = [], []
for N in N_LIST:
    H0, Hc = build_H(N)
    psi0 = start_state(N)
    ref  = reference(H0, Hc, psi0)
    eul  = {s: euler(H0, Hc, psi0, s) for s in STEPS}

    cupy_state = load(f"cupy_state_N{N}.npy")
    dq_state   = load(f"dq_state_N{N}.npy")

    table1.append((
        N,
        np.linalg.norm(ref),
        None if cupy_state is None else rel_err(cupy_state, ref),
        None if dq_state   is None else rel_err(dq_state, ref),
        None if cupy_state is None else rel_err(cupy_state, eul[2000]),
    ))
    table2.append((N, [rel_err(eul[s], ref) for s in STEPS]))

print("TABLE 1: saved results vs the accurate solution")
print(f"{'N':>2} | {'|ref| (=1?)':>12} | {'cupy vs ref':>12} | {'dynamiqs vs ref':>15} | {'cupy vs my Euler':>16}")
for N, nref, e_cp, e_dq, e_me in table1:
    print(f"{N:2d} | {nref:12.8f} | {fmt(e_cp)} | {fmt(e_dq):>15} | {fmt(e_me):>16}")

print("\nTABLE 2: Euler error vs accurate solution as steps increase")
print(f"{'N':>2} | " + " | ".join(f"{s:>9d} steps" for s in STEPS))
for N, errs in table2:
    print(f"{N:2d} | " + " | ".join(f"{e:15.3e}" for e in errs))

TABLE 1: saved results vs the accurate solution
 N |  |ref| (=1?) |  cupy vs ref | dynamiqs vs ref | cupy vs my Euler
 1 |   1.00000000 |    8.795e-04 |       8.795e-04 |        0.000e+00
 2 |   1.00000000 |    1.244e-03 |       1.244e-03 |        4.932e-17
 3 |   1.00000000 |    1.524e-03 |       1.524e-03 |        3.105e-17
 4 |   1.00000000 |    1.759e-03 |       1.759e-03 |        1.047e-16
 5 |   1.00000000 |    1.967e-03 |       1.967e-03 |        2.346e-16

TABLE 2: Euler error vs accurate solution as steps increase
 N |      2000 steps |      4000 steps |      8000 steps |     16000 steps
 1 |       8.795e-04 |       4.182e-04 |       2.040e-04 |       1.007e-04
 2 |       1.244e-03 |       5.915e-04 |       2.885e-04 |       1.425e-04
 3 |       1.524e-03 |       7.245e-04 |       3.534e-04 |       1.745e-04
 4 |       1.759e-03 |       8.366e-04 |       4.081e-04 |       2.015e-04
 5 |       1.967e-03 |       9.354e-04 |       4.562e-04 |       2.253e-04
